# Task 1 - Solucao Guiada (RDS)

Este notebook resume a implementacao da Task 1 em PT-BR.

## Objetivos
- Provisionar uma instancia MySQL no Amazon RDS.
- Carregar o dump `classicmodels` com idempotencia controlada.
- Validar se as tabelas e dados basicos estao presentes.

## Pre-requisitos
- Copiar `.env.example` para `.env` e preencher credenciais.
- Nao incluir segredos diretamente nas celulas.
- Opcional: instalar `jupyter` e `ipykernel` para execucao local.


```mermaid
flowchart LR
  A[provision_rds.py] --> B[Definir RDS_ENDPOINT]
  B --> C[load_data.py]
  C --> D[validate.py]
```


In [ ]:
from pathlib import Path

def detect_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        sentinel = candidate / "scripts" / "provision_rds.py"
        if sentinel.exists():
            return candidate
    raise FileNotFoundError("Nao foi possivel localizar scripts/provision_rds.py")

ROOT = detect_project_root()
ROOT


## 1) Provisionamento

Esta etapa cria (ou reaproveita) a instancia RDS. O script foi feito para ser idempotente e suporta `--dry-run` para revisar parametros sem alterar recursos.


In [ ]:
from IPython.display import Code

Code((ROOT / "scripts" / "provision_rds.py").read_text(encoding="utf-8"), language="python")


In [ ]:
import os
import subprocess

os.chdir(ROOT)
subprocess.run(["python", "scripts/provision_rds.py", "--dry-run"], check=False)


## 2) Carga de dados

Nesta fase, usamos o `mysql` CLI para criar/recriar o schema e carregar o dump SQL completo. Esse caminho evita camadas extras de ORM em um fluxo de bulk load.


In [ ]:
Code((ROOT / "scripts" / "load_data.py").read_text(encoding="utf-8"), language="python")


In [ ]:
os.chdir(ROOT)
subprocess.run(["python", "scripts/load_data.py", "--dry-run"], check=False)


## 3) Validacao

A validacao verifica a presenca das tabelas esperadas e uma contagem minima de registros para reduzir riscos de carga incompleta.


In [ ]:
Code((ROOT / "scripts" / "validate.py").read_text(encoding="utf-8"), language="python")


In [ ]:
os.chdir(ROOT)
subprocess.run(["python", "scripts/validate.py", "--dry-run"], check=False)
